# Bronze to Silver

This notebook turns AEMO's raw actual-demand and P5MIN forecast CSVs into two structured Silver Delta tables in one run.

It reads from the Databricks Volume `/Volumes/workspace/default/aemo_mlops_volume`, where the ingestion notebooks store the Bronze files.

In [ ]:
from pyspark.sql import functions as F

## Parse AEMO files

AEMO CSVs contain several record groups rather than one ordinary header and table. PySpark therefore reads each physical line as text, finds the `I` row that defines the requested group, and keeps only its matching `D` rows.

The exact row pattern matters: `DISPATCH,REGIONSUM` contains actual demand, while `P5MIN,REGIONSOLUTION` contains AEMO's forecast runs. Only the small parser below needs source-specific arguments.

In [ ]:
def parse_aemo(
    path,
    row_pattern,
    time_column,
    forecast_time_column=None,
    filter_runno=False,
):
    # Read lines as text because one AEMO file can contain several record groups.
    raw = spark.read.text(path)

    header_row = (
        raw
        .filter(F.col("value").startswith(f"I,{row_pattern}"))
        .select(F.split("value", ",").alias("fields"))
        .first()
    )

    if header_row is None:
        raise ValueError(f"No I row found for {row_pattern} in {path}")

    header = header_row["fields"]

    # Column positions come from AEMO's I row instead of fragile fixed indexes.
    time_i = header.index(time_column)
    region_i = header.index("REGIONID")
    intervention_i = header.index("INTERVENTION")
    demand_i = header.index("TOTALDEMAND")
    forecast_time_i = (
        header.index(forecast_time_column)
        if forecast_time_column
        else None
    )
    runno_i = header.index("RUNNO") if filter_runno else None

    rows = (
        raw
        .filter(F.col("value").startswith(f"D,{row_pattern}"))
        .withColumn("fields", F.split("value", ","))
        .filter(F.col("fields")[region_i] == "VIC1")
        .filter(F.col("fields")[intervention_i].cast("int") == 0)
    )

    # RUNNO belongs to actual DISPATCH data, not P5MIN REGIONSOLUTION.
    if filter_runno:
        rows = rows.filter(F.col("fields")[runno_i].cast("int") == 1)

    def timestamp_expression(index, alias):
        return F.to_timestamp(
            F.regexp_replace(F.col("fields")[index], '"', ""),
            "yyyy/MM/dd HH:mm:ss",
        ).alias(alias)

    output = []
    if forecast_time_i is not None:
        output.append(timestamp_expression(forecast_time_i, "forecast_time"))

    output.extend([
        timestamp_expression(time_i, "time"),
        F.regexp_replace(F.col("fields")[demand_i], '"', "")
        .cast("double")
        .alias("demand"),
    ])

    return rows.select(*output)

## Actual demand

Monthly, Daily and Current are overlapping delivery mechanisms for the same observed VIC1 dispatch demand. Each source uses `DISPATCH,REGIONSUM`, `SETTLEMENTDATE`, `RUNNO = 1` and `INTERVENTION = 0`, producing the common schema `time | demand`.

In [ ]:
bronze_path = "/Volumes/workspace/default/aemo_mlops_volume/bronze"

monthly = parse_aemo(
    f"{bronze_path}/monthly_uncompressed/*.CSV",
    row_pattern="DISPATCH,REGIONSUM",
    time_column="SETTLEMENTDATE",
    filter_runno=True,
)

daily = parse_aemo(
    f"{bronze_path}/daily_uncompressed/*.CSV",
    row_pattern="DISPATCH,REGIONSUM",
    time_column="SETTLEMENTDATE",
    filter_runno=True,
)

current = parse_aemo(
    f"{bronze_path}/current_uncompressed/*.CSV",
    row_pattern="DISPATCH,REGIONSUM",
    time_column="SETTLEMENTDATE",
    filter_runno=True,
)

## AEMO demand forecast

P5MIN contains repeated forecast runs. `forecast_time` records when AEMO produced a forecast; `time` records the future five-minute interval being predicted. Both timestamps must remain because the same future interval appears in several forecast runs.

Archive and Current forecast files are read together. P5MIN has no `RUNNO` filter, and every available forecast horizon is retained.

In [ ]:
forecast = parse_aemo(
    [
        f"{bronze_path}/forecast/archive_uncompressed/*.CSV",
        f"{bronze_path}/forecast/current_uncompressed/*.CSV",
    ],
    row_pattern="P5MIN,REGIONSOLUTION",
    time_column="INTERVAL_DATETIME",
    forecast_time_column="RUN_DATETIME",
)

## Create structured views

These four views are the boundary between AEMO-specific file parsing and ordinary structured processing. From this point onward, straightforward Spark SQL can build the two Silver tables.

In [ ]:
monthly.createOrReplaceTempView("monthly")
daily.createOrReplaceTempView("daily")
current.createOrReplaceTempView("current")
forecast.createOrReplaceTempView("forecast")

## Build the actual Silver table

The actual sources overlap. The existing rule is preserved: when a timestamp appears more than once, prefer Current, then Daily, then Monthly because fresher publications may contain corrections.

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.demand_vic_5min
USING DELTA
AS

WITH all_actual AS (
    SELECT time, demand, 1 AS priority FROM monthly
    UNION ALL
    SELECT time, demand, 2 AS priority FROM daily
    UNION ALL
    SELECT time, demand, 3 AS priority FROM current
),

ranked AS (
    SELECT
        time,
        demand,
        ROW_NUMBER() OVER (
            PARTITION BY time
            ORDER BY priority DESC
        ) AS row_number
    FROM all_actual
)

SELECT time, demand
FROM ranked
WHERE row_number = 1

## Build the forecast Silver table

The forecast key is `(forecast_time, time)`, not `time` alone. Exact duplicate rows from overlapping Archive and Current files are removed without inventing a source-priority rule.

Before writing Silver, the validation below exposes any key that has different demand values. The pipeline stops rather than silently choosing one.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW forecast_conflicts AS
SELECT
    forecast_time,
    time,
    COUNT(DISTINCT demand) AS demand_values
FROM forecast
GROUP BY forecast_time, time
HAVING COUNT(DISTINCT demand) > 1

In [ ]:
%sql
SELECT *
FROM forecast_conflicts
ORDER BY forecast_time, time

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 0,
    'Conflicting P5MIN demand values found for the same forecast_time and time'
) AS forecast_conflict_check
FROM forecast_conflicts

In [ ]:
%sql
CREATE OR REPLACE TABLE workspace.default.aemo_demand_forecast_vic_5min
USING DELTA
AS
SELECT DISTINCT
    forecast_time,
    time,
    demand
FROM forecast

## Validation

The actual table should contain one demand value per five-minute timestamp. For forecasts, one `forecast_time` should contain several future intervals, and the same future `time` should appear under several forecast runs. That repeated structure is expected and must be preserved.

In [ ]:
%sql
SELECT *
FROM workspace.default.demand_vic_5min
ORDER BY time
LIMIT 100

In [ ]:
%sql
SELECT *
FROM workspace.default.aemo_demand_forecast_vic_5min
ORDER BY forecast_time, time
LIMIT 100

In [ ]:
%sql
SELECT
    forecast_time,
    COUNT(DISTINCT time) AS future_intervals
FROM workspace.default.aemo_demand_forecast_vic_5min
GROUP BY forecast_time
ORDER BY forecast_time DESC
LIMIT 20

In [ ]:
%sql
SELECT
    time,
    COUNT(DISTINCT forecast_time) AS forecast_vintages
FROM workspace.default.aemo_demand_forecast_vic_5min
GROUP BY time
HAVING COUNT(DISTINCT forecast_time) > 1
ORDER BY time DESC
LIMIT 20